# Lecția 11 - Model Context Protocol (MCP)

**Model Context Protocol (MCP)** este un standard deschis care permite agenților să descopere și să utilizeze dinamic unelte, resurse și surse de date în timpul execuției. În loc să codifice uneltele direct într-un agent, MCP permite agenților să se conecteze la servere externe care expun capabilități la cerere.

În această lecție, vei învăța:
- Ce este MCP și de ce contează pentru sistemele de agenți
- Cum funcționează arhitectura client-server MCP
- Cum să construiești agenți care folosesc descoperirea uneltelor în stil MCP


## Configurare

**Cerințe prealabile:**
- Proiect Azure AI Foundry cu un model implementat
- Rulați `az login` pentru autentificare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ce este Protocolul Contextului Modelului (MCP)?

MCP definește o modalitate standard pentru ca agenții AI să descopere și să interacționeze cu instrumente și surse de date externe:

- **MCP Server**: Expune instrumente, resurse și prompturi printr-un protocol standard
- **MCP Client**: Runtime-ul agentului care se conectează la servere și descoperă capabilitățile disponibile
- **Dynamic Discovery**: Agenții nu au nevoie de unelte codificate în mod static — descoperă ce este disponibil în timpul execuției

Acest lucru este foarte util pentru construirea de sisteme de agenți extensibile, unde pot fi adăugate noi capabilități fără a modifica codul agentului.


## Cum funcționează MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agentul (clientul MCP) se conectează la un server MCP
2. Serverul răspunde cu o listă de instrumente disponibile și schemele acestora
3. Agentul poate apoi apela orice instrument descoperit în timpul raționamentului său
4. Rezultatele se întorc prin același protocol


## Simularea descoperirii instrumentelor MCP

Deoarece un server MCP real necesită un proces de server în funcțiune, vom demonstra acest model folosind funcții `@tool` care simulează ceea ce ar furniza un serviciu de cazare conectat la MCP.

În producție, aceste instrumente ar fi descoperite dinamic de către un server MCP, în loc să fie definite local.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Construirea unui agent cu instrumente în stil MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP în producție

Într-un mediu de producție, MCP permite tipare puternice:

- **Descoperire dinamică a instrumentelor**: Agenții se conectează la serverele MCP și descoperă instrumentele în timpul rulării
- **Arhitectură decuplată**: Furnizorii de instrumente pot face actualizări independent de agent
- **Partajare între organizații**: Echipele pot expune capabilități prin servere MCP pe care orice agent le poate folosi
- **Suport Microsoft Agent Framework**: MAF oferă suport încorporat pentru clientul MCP prin integrarea `mcp`

Pentru a folosi un server MCP real cu MAF, vă puteți conecta prin `hosted_mcp_tool()` sau prin integrarea clientului MCP.

**Aflați mai multe:**
- [Specificația MCP](https://modelcontextprotocol.io/)
- [Suport MCP pentru Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Rezumat

În această lecție, ai învățat:
- **MCP** este un standard deschis pentru descoperirea dinamică a instrumentelor între agenți și furnizorii de instrumente
- Această **arhitectură client-server** permite agenților să descopere capabilități în timpul execuției
- MCP permite **sisteme de agenți extensibile, decuplate** în care instrumentele pot fi adăugate fără modificări de cod
- Microsoft Agent Framework oferă **suport MCP integrat** pentru utilizare în producție


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare de responsabilitate**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original, în limba sa nativă, trebuie considerat sursa autorizată. Pentru informații critice, se recomandă o traducere profesională realizată de un traducător uman. Nu ne asumăm nicio răspundere pentru eventualele neînțelegeri sau interpretări greșite care pot rezulta din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
